In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML and Metrics
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, recall_score, f1_score, roc_auc_score

# Advanced Generative Modeling (Replacing SMOTE with SDV as per proposal)
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata

import warnings
warnings.filterwarnings('ignore')

In [3]:
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata

In [4]:
# Load dataset
df = pd.read_csv("") 

X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split ensures the minority class is represented in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Shape: {X_train.shape}")
print(f"Class Distribution:\n{y_train.value_counts()}")

FileNotFoundError: [Errno 2] No such file or directory: ''

In [ ]:
import sys
print(sys.executable)

c:\Users\Ainee\AppData\Local\Programs\Python\Python311\python.exe


In [ ]:
def evaluate_performance(model, X_tst, y_tst, label="Model"):
    y_pred = model.predict(X_tst)
    y_proba = model.predict_proba(X_tst)[:, 1]
    
    metrics = {
        "Recall": recall_score(y_tst, y_pred),
        "F1-Score": f1_score(y_tst, y_pred),
        "AUC-ROC": roc_auc_score(y_tst, y_proba)
    }
    
    print(f"\n--- {label} Results ---")
    print(classification_report(y_tst, y_pred))
    return metrics

# Train Baseline
print("Training Baseline Model...")
baseline_model = RandomForestClassifier(n_estimators=100, random_state=42)
baseline_model.fit(X_train, y_train)
baseline_metrics = evaluate_performance(baseline_model, X_test, y_test, "BASELINE")

In [ ]:
def identify_errors(model, X, y):
    """Identifies 'False Negatives' - actual fraud missed by the model."""
    preds = model.predict(X)
    error_mask = (y == 1) & (preds == 0)
    return X[error_mask]

def generate_targeted_data(X_tr, y_tr, target_rows=200):
    """Generates synthetic data using the Gaussian Copula model from SDV."""
    combined = X_tr.copy()
    combined['Class'] = y_tr
    
    # Detect metadata for the generative model
    metadata = Metadata.detect_from_dataframe(combined)
    
    # Initialize and fit the Synthesizer
    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(combined)
    
    # Generate new synthetic samples
    synthetic_samples = synthesizer.sample(num_rows=target_rows)
    
    X_aug = pd.concat([X_tr, synthetic_samples.drop('Class', axis=1)], axis=0)
    y_aug = pd.concat([y_tr, synthetic_samples['Class']], axis=0)
    
    return X_aug, y_aug

In [ ]:
X_loop, y_loop = X_train.copy(), y_train.copy()
best_recall = baseline_metrics["Recall"]
history = [best_recall]

print("Starting Adaptive Loop...")

for i in range(3):
    # 1. Analyze current model errors
    errors = identify_errors(baseline_model, X_loop, y_loop)
    
    if len(errors) > 0:
        print(f"\nIteration {i+1}: Found {len(errors)} errors. Generating targeted data...")
        
        # 2. Adaptive Augmentation
        X_loop, y_loop = generate_targeted_data(X_loop, y_loop, target_rows=200)
        
        # 3. Retrain
        loop_model = RandomForestClassifier(n_estimators=100, random_state=42)
        loop_model.fit(X_loop, y_loop)
        
        # 4. Evaluate
        current_metrics = evaluate_performance(loop_model, X_test, y_test, f"ITERATION {i+1}")
        history.append(current_metrics["Recall"])
        
        # Check against Proposal Success Criteria (10% improvement)
        improvement = current_metrics["Recall"] - baseline_metrics["Recall"]
        if improvement >= 0.10:
            print(f"!!! Success Criteria Met: {improvement*100:.2f}% improvement achieved.")
            break
    else:
        break

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(len(history)), history, marker='o', linestyle='--', color='b')
plt.title("Toolkit Progress: Recall Improvement over Iterations")
plt.xlabel("Iteration (0 = Baseline)")
plt.ylabel("Recall Score")
plt.grid(True)
plt.show()

print(f"Final Recall: {history[-1]:.4f}")
print(f"Total Improvement: {(history[-1] - history[0])*100:.2f}%")